In [ ]:
# User Input Text
text_input = input("You: ")

## Test Ollama

In [8]:
# Ollama LLM
import ollama

response = ollama.chat(
    model="llama3.2:3b",
    messages=[{"role": "user", "content": "Say hello and tell me you're running locally."}],
    think=False
)
print(response["message"])

role='assistant' content="Hello! I'm a large language model, but I don't have a physical presence, so I won't be able to run in the classical sense. However, I'm here and ready to help answer any questions or provide assistance you need! How can I assist you today?" thinking=None images=None tool_name=None tool_calls=None


## Test ChromaDB

In [13]:
! pip install chromadb
! ollama pull nomic-embed-text

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


In [1]:
import chromadb
import ollama

# Create Chroma Client
chroma_client = client = chromadb.PersistentClient(path="../data/chroma_db")

# Create collections ( store embeddings, documents, and additional metadata )
collections = chroma_client.get_or_create_collection(name="jarvis_knowledge")

# Add a few test documents
docs = [
    "JARVIS is a personal offline AI assistant built with Ollama and ChromaDB.",
    "The user's name is Jason, also known as Phong Dinh.",
    "JARVIS can open apps like Notion, VS Code, and Brave, etc.",
]

# Helper to get embeddings from Ollama
def get_embedding(text):
    response = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return response['embedding']

collections.add(
    documents=docs,
    embeddings=[get_embedding(text) for text in docs],
    ids=[f"doc_{i}" for i in range(len(docs))]
)

print("Documents added successfully")

query = "What can JARVIS do with apps?"
query_embedding = get_embedding(query)

results = collections.query(
    query_embeddings=[query_embedding],
    n_results=2
)

print(results["documents"])

Documents added successfully
[['JARVIS can open apps like Notion, VS Code, and Brave, etc.', 'JARVIS is a personal offline AI assistant built with Ollama and ChromaDB.']]


## Build simple retrieval system + test latency

In [3]:
def ask_JARVIS(query, n_results = 2):
    query_embedding = get_embedding(query)
    results = collections.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    context = '\n'.join(results['documents'][0])

    # Prompt
    PROMPT = f"""
    Use the following context to answer the question. If the context doesn't contain the answer, say no.
    Context: {context}
    Question: {query}
    Answer: 
    """

    response = ollama.chat(
        model="llama3.2:3b",
        messages= [{"role": "user", "content": PROMPT}],
        think=False
    )
    return response['message']['content']

print(ask_JARVIS(query, 2))



JARVIS can open apps.


## Test Whisper to turn speech to text

In [ ]:
# User Input Voice
import sounddevice as sd
import numpy as np
import keyboard

duration = 5
SAMPLE_RATE = 16000
TRIGGER_KEY="Esc"

def record_while_held(key=TRIGGER_KEY):
    print(f"Hold {key} to talk...")
    keyboard.wait(key)
    print("Recording...(release to stop)")

    frames = []
    def callback(indata, frame_count, time_info, status):
        frames.append(indata.copy())
    
    with sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="float32", callback=callback):
        while keyboard.is_pressed(key):
            sd.sleep(50)

    print("Stopped recording.")
    audio = np.concatenate(frames, axis=0)
    return audio.flatten()

#Test
audio_data = record_while_held()
print(f"Captured {len(audio_data)} samples {len(audio_data) / SAMPLE_RATE:.2f} seconds.")


: 

In [ ]:
# Faster-Whisper
from faster_whisper import WhisperModel

model_size = "small"
model = WhisperModel(model_size, device="cpu", compute_type="int8")

segments, info = model.transcribe(audio_data, language='en', beam_size=5)

print("Detected language '%s' with probability %f" % (info.language, info.language_probability))

for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

user_text = " ".join([seg.text for seg in segments])